In [8]:
import pandas as pd
import cv2

frames1 = pd.read_csv("kaggle_dataset/labels_train.csv")
frames2 = pd.read_csv("kaggle_dataset/labels_val.csv")
frames3 = pd.read_csv("kaggle_dataset/labels_trainval.csv")

frames = pd.concat([frames1, frames2, frames3])

frames = frames[frames["class_id"] == 1]




print(frames.shape)



(246628, 6)


In [9]:

frames = frames.sample(14000, random_state=42)


frames = (
    frames.groupby("frame")[["xmin", "xmax", "ymin", "ymax"]]
    .apply(lambda x: x.values.tolist())
    .reset_index(name="boxes")
)

print(frames.shape)


(9669, 2)


In [10]:
def recalculate_bounding_box(row):
    boxes = row["boxes"]
    scaled_boxes = []
    image = cv2.imread("kaggle_dataset/images/" + row['frame'])
    h, w, _ = image.shape

    for i, box in enumerate(boxes):
        xmin = box[0]
        xmax = box[1]
        ymin = box[2]
        ymax = box[3]

        
        new_xmin = int(xmin * (128 / w))
        new_ymin = int(ymin * (128 / h))
        new_xmax = int(xmax * (128 / w))
        new_ymax = int(ymax * (128 / h))

        scaled_boxes.append([new_xmin, new_xmax, new_ymin, new_ymax])

    return scaled_boxes

# test aşamasın iou için en büyük nesneyi temel alalım. Yani ilk sırada olacak olan
def calculate_box_area(box):
    xmin, xmax, ymin, ymax = box

    x = xmax-xmin
    y = ymax-ymin

    return x*y

for i, row in frames.iterrows():
    boxes = recalculate_bounding_box(row)
    
    unique = []
    for item in boxes:
        if item not in unique:
            unique.append(item)


    boxes.sort(key=calculate_box_area)
    frames.at[i,"boxes"] = unique

print(frames["boxes"])



0       [[99, 121, 50, 64], [51, 55, 61, 66]]
1                          [[47, 51, 60, 64]]
2                          [[47, 50, 61, 65]]
3                          [[46, 48, 61, 65]]
4                        [[111, 120, 57, 61]]
                        ...                  
9664                       [[32, 35, 61, 64]]
9665                       [[34, 37, 60, 64]]
9666                       [[60, 66, 59, 68]]
9667                       [[30, 36, 60, 71]]
9668                     [[102, 127, 54, 89]]
Name: boxes, Length: 9669, dtype: object


In [11]:
import numpy as np

# 70 train 15 val 15 test

frames_train = frames.sample(frac=0.7, random_state=46)
frames_val = frames[~frames.index.isin(frames_train.index)].sample(frac=0.5, random_state=46)
frames_test = frames[~frames.index.isin(pd.concat([frames_train, frames_val]).index)]

np.save("train_frames.npy", frames_train.to_numpy(dtype=object))
np.save("val_frames.npy", frames_val.to_numpy(dtype=object))
np.save("test_frames.npy", frames_test.to_numpy(dtype=object))

print(len(frames_train))
print(len(frames_val))
print(len(frames_test))



6768
1450
1451


In [12]:
import os

os.makedirs("images", exist_ok= True)

for i, row in frames.iterrows():
    image = cv2.imread("kaggle_dataset/images/" + row['frame'], cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, (128,128))
    cv2.imwrite("images/" + row['frame'], image)

